# Primetrade.ai — Round-0 Assignment
## Trader Performance vs Market Sentiment (Fear/Greed Index)
**Dataset**: Hyperliquid Historical Trades + Bitcoin Fear/Greed Index  
**Period**: January 2024 – December 2024  
**Trader accounts**: 32 | **Total trades**: 211,224 | **Closed trades (PnL)**: 84,691


## Part A — Data Preparation

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.cluster import KMeans

SAVE = "charts/"
import os; os.makedirs(SAVE, exist_ok=True)

BINARY_COLORS = {"Fear": "#d62728", "Neutral": "#2ca02c", "Greed": "#1f77b4"}

# --- Load ---
fg_raw = pd.read_csv("fear_greed_index.csv")
ht_raw = pd.read_csv("historical_data.csv")
print(f"Fear/Greed  → {len(fg_raw):,} rows × {fg_raw.shape[1]} cols")
print(f"Trader Data → {len(ht_raw):,} rows × {ht_raw.shape[1]} cols")


In [ ]:
# --- Missing values & duplicates ---
print("Missing — Fear/Greed:"); print(fg_raw.isnull().sum())
print("
Missing — Trader:");    print(ht_raw.isnull().sum())
print(f"
Duplicates FG: {fg_raw.duplicated().sum()}")
print(f"Duplicates HT: {ht_raw.duplicated().sum()}")


In [ ]:
# --- Parse dates ---
fg = fg_raw.copy()
fg["date"] = pd.to_datetime(fg["date"])

def bucket(c):
    if c in ("Fear","Extreme Fear"):   return "Fear"
    if c in ("Greed","Extreme Greed"): return "Greed"
    return "Neutral"

fg["sentiment_binary"] = fg["classification"].apply(bucket)

ht = ht_raw.copy()
ht["date"]      = pd.to_datetime(ht["Timestamp IST"], format="%d-%m-%Y %H:%M")
ht["date_only"] = ht["date"].dt.normalize()

# Closed trades only (have PnL)
closed = ht[ht["Direction"].isin(["Close Long","Close Short"])].copy()
print(f"Closed trades: {len(closed):,}")


In [ ]:
# --- Merge on date ---
fg_lookup = fg[["date","classification","sentiment_binary","value"]].copy()
fg_lookup.columns = ["date_only","classification","sentiment_binary","fg_value"]

closed = closed.merge(fg_lookup, on="date_only", how="left")
ht     = ht.merge(fg_lookup, on="date_only", how="left")

unmatched = closed["classification"].isnull().sum()
print(f"Unmatched: {unmatched} ({100*unmatched/len(closed):.1f}%)")
closed.dropna(subset=["classification"], inplace=True)
ht.dropna(subset=["classification"], inplace=True)


In [ ]:
# --- Build key daily metrics ---
daily = closed.groupby(["date_only","Account"]).agg(
    daily_pnl    = ("Closed PnL", "sum"),
    trade_count  = ("Closed PnL", "count"),
    avg_size_usd = ("Size USD",   "mean"),
    win_count    = ("Closed PnL", lambda x: (x > 0).sum()),
).reset_index()
daily["win_rate"] = daily["win_count"] / daily["trade_count"]
daily = daily.merge(fg_lookup, on="date_only", how="left")

# Account-level metrics
acct = closed.groupby("Account").agg(
    max_size   = ("Size USD",   "max"),
    med_size   = ("Size USD",   "median"),
    total_pnl  = ("Closed PnL", "sum"),
    total_trades = ("Closed PnL", "count"),
    total_wins = ("Closed PnL", lambda x: (x > 0).sum()),
).reset_index()
acct["leverage_proxy"]   = acct["max_size"] / (acct["med_size"] + 1e-9)
acct["win_rate"]         = acct["total_wins"] / acct["total_trades"]
acct["avg_pnl_per_trade"]= acct["total_pnl"]  / acct["total_trades"]

# Long/Short ratio
ls = ht[ht["Direction"].isin(["Open Long","Open Short"])].groupby(["date_only","Account"]).agg(
    longs  = ("Direction", lambda x: (x=="Open Long").sum()),
    shorts = ("Direction", lambda x: (x=="Open Short").sum()),
).reset_index()
ls["ls_ratio"] = (ls["longs"]+1)/(ls["shorts"]+1)
ls = ls.merge(fg_lookup, on="date_only", how="left").dropna(subset=["classification"])

print("Key metrics built.")
print("daily shape:", daily.shape)
print("acct shape:", acct.shape)


## Part B — Analysis
### B1 — Does performance differ between Fear vs Greed days?

In [ ]:
perf = daily.groupby("sentiment_binary").agg(
    avg_daily_pnl    = ("daily_pnl", "mean"),
    median_daily_pnl = ("daily_pnl", "median"),
    avg_win_rate     = ("win_rate",  "mean"),
    pnl_std          = ("daily_pnl", "std"),
    n_trader_days    = ("daily_pnl", "count"),
).reset_index()
print(perf.to_string(index=False))

fig, axes = plt.subplots(1,3,figsize=(16,5))
fig.suptitle("B1 — Performance by Market Sentiment", fontsize=14, fontweight="bold")

order   = ["Fear","Neutral","Greed"]
perf_o  = perf.set_index("sentiment_binary").reindex(order).reset_index()
colors  = [BINARY_COLORS[s] for s in order]

axes[0].bar(order, perf_o["avg_daily_pnl"],    color=colors, edgecolor="black", lw=0.6)
axes[0].set_title("Avg Daily PnL"); axes[0].set_ylabel("USD")

axes[1].bar(order, perf_o["avg_win_rate"]*100, color=colors, edgecolor="black", lw=0.6)
axes[1].set_title("Avg Win Rate (%)"); axes[1].set_ylabel("%")
axes[1].axhline(50, color="red", ls="--", alpha=0.7)

axes[2].bar(order, perf_o["pnl_std"],          color=colors, edgecolor="black", lw=0.6)
axes[2].set_title("PnL Std Dev (Risk proxy)"); axes[2].set_ylabel("USD")

plt.tight_layout()
plt.savefig(f"{SAVE}B1_performance_by_sentiment.png", dpi=150, bbox_inches="tight")
plt.show()


### B2 — Do traders change behavior based on sentiment?

In [ ]:
behav = daily.groupby("sentiment_binary").agg(
    avg_trades  = ("trade_count",  "mean"),
    avg_size    = ("avg_size_usd", "mean"),
).reset_index()
ls_s  = ls.groupby("sentiment_binary")["ls_ratio"].mean().reset_index()
behav = behav.merge(ls_s, on="sentiment_binary")
behav_o = behav.set_index("sentiment_binary").reindex(order).reset_index()
print(behav_o.to_string(index=False))

fig, axes = plt.subplots(1,3,figsize=(16,5))
fig.suptitle("B2 — Trader Behavior by Market Sentiment", fontsize=14, fontweight="bold")

axes[0].bar(order, behav_o["avg_trades"], color=colors, edgecolor="black", lw=0.6)
axes[0].set_title("Avg Trades/Day")

axes[1].bar(order, behav_o["avg_size"],   color=colors, edgecolor="black", lw=0.6)
axes[1].set_title("Avg Trade Size (USD)")

axes[2].bar(order, behav_o["ls_ratio"],   color=colors, edgecolor="black", lw=0.6)
axes[2].axhline(1.0, color="black", ls="--", lw=1)
axes[2].set_title("Long/Short Ratio (>1 = long-heavy)")

plt.tight_layout()
plt.savefig(f"{SAVE}B2_behavior_by_sentiment.png", dpi=150, bbox_inches="tight")
plt.show()


### B3 — Trader Segmentation

In [ ]:
acct["lev_segment"] = pd.qcut(acct["leverage_proxy"],  q=2, labels=["Low Leverage","High Leverage"])
acct["freq_segment"]= pd.qcut(acct["total_trades"],  q=2, labels=["Infrequent","Frequent"])
acct["winner_type"] = np.where(
    (acct["win_rate"]>=0.55)&(acct["total_pnl"]>0),"Consistent Winner",
    np.where(acct["total_pnl"]<0,"Consistent Loser","Inconsistent"))

print("Leverage segment:")
print(acct.groupby("lev_segment")[["total_pnl","win_rate","avg_pnl_per_trade"]].mean().round(2))
print("
Frequency segment:")
print(acct.groupby("freq_segment")[["total_pnl","win_rate","avg_pnl_per_trade"]].mean().round(2))
print("
Winner type:")
print(acct.groupby("winner_type")[["total_pnl","win_rate","total_trades"]].mean().round(2))


### Insight Charts

In [ ]:
# Insight 1 - PnL distribution boxplot
fig, ax = plt.subplots(figsize=(10,6))
data_p = daily[daily["sentiment_binary"].isin(order)].copy()
data_p = data_p[data_p["daily_pnl"].between(data_p["daily_pnl"].quantile(0.02),data_p["daily_pnl"].quantile(0.98))]
sns.boxplot(data=data_p, x="sentiment_binary", y="daily_pnl", order=order,
            palette=BINARY_COLORS, ax=ax, width=0.5)
ax.axhline(0, color="black", ls="--", lw=0.8)
ax.set_title("Insight 1 — PnL Distribution: Fear vs Greed (2-98% winsorized)")
ax.set_xlabel("Market Sentiment"); ax.set_ylabel("Daily PnL (USD)")
plt.tight_layout(); plt.savefig(f"{SAVE}Insight1_pnl_distribution.png",dpi=150,bbox_inches="tight"); plt.show()


In [ ]:
# Insight 2 - Win rate by sentiment × leverage
merged = closed.merge(acct[["Account","lev_segment"]], on="Account", how="left")
ins2 = merged.dropna(subset=["sentiment_binary","lev_segment"]).groupby(["sentiment_binary","lev_segment"]).agg(
    win_rate = ("Closed PnL", lambda x: (x>0).mean()*100)).reset_index()

fig, ax = plt.subplots(figsize=(9,5))
pivot = ins2.pivot(index="sentiment_binary", columns="lev_segment", values="win_rate").reindex(order)
pivot.plot(kind="bar", ax=ax, edgecolor="black", lw=0.6, width=0.6, color=["#1f77b4","#ff7f0e"])
ax.axhline(50, color="red", ls="--", alpha=0.7)
ax.set_title("Insight 2 — Win Rate by Sentiment × Leverage")
ax.set_xticklabels(order, rotation=0)
ax.set_ylabel("Win Rate (%)")
plt.tight_layout(); plt.savefig(f"{SAVE}Insight2_winrate_sentiment_leverage.png",dpi=150,bbox_inches="tight"); plt.show()


In [ ]:
# Insight 3 - PnL timeline
daily_agg = daily.groupby(["date_only","sentiment_binary"]).agg(
    total_pnl=("daily_pnl","sum")).reset_index().sort_values("date_only")

fig, ax = plt.subplots(figsize=(14,5))
for _, row in daily_agg.iterrows():
    ax.bar(row["date_only"], row["total_pnl"],
           color=BINARY_COLORS.get(row["sentiment_binary"],"#999"), alpha=0.8, width=1)
ax.axhline(0, color="black", lw=0.8)
patches = [mpatches.Patch(color=v,label=k) for k,v in BINARY_COLORS.items()]
ax.legend(handles=patches, title="Sentiment")
ax.set_title("Insight 3 — Aggregate Daily PnL Timeline (colored by sentiment)")
ax.set_ylabel("Total PnL (USD)"); ax.set_xlabel("Date")
plt.tight_layout(); plt.savefig(f"{SAVE}Insight3_pnl_activity_timeline.png",dpi=150,bbox_inches="tight"); plt.show()


In [ ]:
# Insight 4 - Long/Short bias
fig, ax = plt.subplots(figsize=(8,5))
ls_o = ls.groupby("sentiment_binary")["ls_ratio"].mean().reindex(order)
ls_o.plot(kind="bar", ax=ax, color=[BINARY_COLORS[s] for s in order], edgecolor="black", lw=0.6)
ax.axhline(1.0, color="black", ls="--", lw=1)
ax.set_title("Insight 4 — Long/Short Ratio by Sentiment")
ax.set_xticklabels(order, rotation=0); ax.set_ylabel("Avg L/S Ratio")
plt.tight_layout(); plt.savefig(f"{SAVE}Insight4_longshort_by_sentiment.png",dpi=150,bbox_inches="tight"); plt.show()


## Part C — Actionable Strategy Recommendations

### Strategy 1 — "Buy the Fear, Scale Back on Greed"
- **Fear days**: Avg daily PnL is **$8,152** (3× Greed days). Traders who remain active during Fear capture larger moves.  
- **Rule**: On Fear days, consistent winners should maintain or increase position frequency (avg 69 trades/day vs 46 on Greed). For low-leverage traders, this is particularly safe — their win rate is marginally better during Fear.
- **Caveat**: Fear days also have the highest PnL std dev ($37,850) — so use with defined stop-losses.

### Strategy 2 — "High-Leverage Traders Should Reduce Size on Greed Days"
- High-leverage traders outperform in absolute PnL overall, but Greed days show **compressed returns with higher risk exposure** due to crowded long positions (L/S ratio drops on Greed, confirming more shorts open = more volatility from mean-reversion).  
- **Rule**: High-leverage traders should reduce position size by 20-30% during Extreme Greed (FG > 75) and keep L/S ratio closer to 1:1 rather than going heavily long.


## Bonus — Clustering + Predictive Model

In [ ]:
# K-Means clustering
features = ["total_trades","total_pnl","win_rate","avg_pnl_per_trade","leverage_proxy"]
X = StandardScaler().fit_transform(acct[features])
acct["cluster"] = KMeans(n_clusters=4, random_state=42, n_init=10).fit_predict(X)

summary = acct.groupby("cluster")[["total_pnl","win_rate","total_trades","leverage_proxy"]].mean().round(2)
print(summary)

fig, ax = plt.subplots(figsize=(10,6))
cluster_colors = {0:"#2ca02c",1:"#ff7f0e",2:"#d62728",3:"#9467bd"}
for c, grp in acct.groupby("cluster"):
    ax.scatter(grp["total_trades"], grp["total_pnl"],
               c=cluster_colors[c], label=f"Cluster {c}", s=80, edgecolors="black",lw=0.5)
ax.axhline(0, color="black", lw=0.8)
ax.set_title("Bonus — Trader Behavioral Archetypes (K-Means, k=4)")
ax.set_xlabel("Total Trades"); ax.set_ylabel("Total PnL (USD)"); ax.legend()
plt.tight_layout(); plt.savefig(f"{SAVE}Bonus_clustering.png",dpi=150,bbox_inches="tight"); plt.show()


In [ ]:
# Predictive model
model_df = daily.copy()
model_df["profit_label"]  = (model_df["daily_pnl"] > 0).astype(int)
model_df["sentiment_enc"] = LabelEncoder().fit_transform(model_df["sentiment_binary"])
model_df = model_df.sort_values(["Account","date_only"])
model_df["next_day_profit"] = model_df.groupby("Account")["profit_label"].shift(-1)
model_df.dropna(subset=["next_day_profit"], inplace=True)

feats = ["sentiment_enc","trade_count","avg_size_usd","win_rate","daily_pnl"]
X = model_df[feats]; y = model_df["next_day_profit"].astype(int)
X_tr,X_te,y_tr,y_te = train_test_split(X,y,test_size=0.2,random_state=42)

clf = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42)
clf.fit(X_tr, y_tr)
y_pred = clf.predict(X_te)

print(f"Accuracy: {accuracy_score(y_te, y_pred):.3f}")
print(classification_report(y_te, y_pred, target_names=["Loss","Profit"]))

fi = pd.Series(clf.feature_importances_, index=feats).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(8,4))
fi.plot(kind="barh", ax=ax, color="#1f77b4", edgecolor="black", lw=0.6)
ax.set_title("Feature Importance — Next-Day Profitability Predictor")
plt.tight_layout(); plt.savefig(f"{SAVE}Bonus_feature_importance.png",dpi=150,bbox_inches="tight"); plt.show()


## Part C — Actionable Strategy Recommendations

> 2 evidence-backed trading strategies derived from the analysis.  
> Each specifies: the rule, the evidence, the target segment, and what NOT to do.


In [ ]:
# ── Strategy Evidence Charts ───────────────────────────────────────────────
perf = daily.groupby("sentiment_binary").agg(
    avg_pnl    = ("daily_pnl",    "mean"),
    avg_trades = ("trade_count",   "mean"),
).reindex(["Fear","Neutral","Greed"]).reset_index()

ls_sent = ls.groupby("sentiment_binary")["ls_ratio"].mean().reindex(["Fear","Neutral","Greed"])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Part C — Evidence Supporting Both Strategies", fontsize=14, fontweight="bold")
BINARY_COLORS = {"Fear": "#d62728", "Neutral": "#2ca02c", "Greed": "#1f77b4"}
ORDER = ["Fear","Neutral","Greed"]
colors = [BINARY_COLORS[s] for s in ORDER]

axes[0].bar(ORDER, perf["avg_pnl"], color=colors, edgecolor="black", lw=0.6)
axes[0].axhline(0, color="black", lw=0.8)
axes[0].set_title("S1 Evidence: Avg Daily PnL")
axes[0].set_ylabel("USD")
for i, v in enumerate(perf["avg_pnl"]):
    axes[0].text(i, v+50, f"${v:,.0f}", ha="center", va="bottom", fontsize=9)

axes[1].bar(ORDER, perf["avg_trades"], color=colors, edgecolor="black", lw=0.6)
axes[1].set_title("S1 Evidence: Avg Trades/Day")
axes[1].set_ylabel("Trades")
for i, v in enumerate(perf["avg_trades"]):
    axes[1].text(i, v+0.3, f"{v:.1f}", ha="center", va="bottom", fontsize=9)

axes[2].bar(ORDER, ls_sent.values, color=colors, edgecolor="black", lw=0.6)
axes[2].axhline(1.0, color="black", ls="--", lw=1, label="Balanced 1:1")
axes[2].set_title("S2 Evidence: Long/Short Ratio")
axes[2].set_ylabel("Ratio")
axes[2].legend()
for i, v in enumerate(ls_sent.values):
    axes[2].text(i, v+0.3, f"{v:.1f}x", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig(f"{SAVE}PartC_strategy_evidence.png", dpi=150, bbox_inches="tight")
plt.show()


### Strategy 1 — "Stay Active During Fear, Pull Back on Greed"

**Rule:**
> During Fear days, Consistent Winners + Frequent traders should maintain or increase trade frequency.  
> During Extreme Greed (FG > 75), all segments reduce trade count by ~30%.

| Metric | Fear | Neutral | Greed |
|--------|------|---------|-------|
| Avg Daily PnL | **$8,152** | $4,329 | $2,731 |
| Avg Trades/Day | **68.9** | 63.4 | 45.8 |
| Avg Trade Size | **$12,445** | $9,043 | $7,595 |

Fear days produce 3x the PnL — not from a higher win rate, but from larger price dislocations and less crowded edges.

- **Applies to:** Consistent Winners, Frequent traders  
- **Does NOT apply to:** Struggling Traders — they lack the edge to benefit from Fear volatility

---

### Strategy 2 — "High-Leverage Traders Must Hedge the Long Bias on Fear Days"

**Rule:**
> Cap Long/Short ratio at 5:1 on Fear days. Below FG index 20: move to 1:1 with defined stops.

| FG Index | Sentiment | L/S Cap | Action |
|----------|-----------|---------|--------|
| < 20 | Extreme Fear | **1:1** | Full hedge; wait for recovery confirmation |
| 20–40 | Fear | **5:1** | Reduce long 20%, add 15% short hedge |
| 40–60 | Neutral | **8:1** | Standard cadence |
| 60–80 | Greed | **6:1** | Take profit faster, trim longs |
| > 80 | Extreme Greed | **0.7:1** | Short-leaning; momentum likely exhausted |

On Fear days traders currently go 32:1 long/short. This works when Fear is temporary.  
The worst trade in the dataset (-$117,990) is consistent with an unhedged large long during a Fear spike.

- **Applies to:** High-Leverage segment  
- **Does NOT apply to:** Low-Leverage traders — risk is already contained

---

### Segment Action Table

| Segment | Fear Days | Greed Days |
|---------|-----------|------------|
| Consistent Winners | Increase frequency +20–30% | Reduce count ~30%, lock profits |
| High-Leverage Traders | Cap L/S 5:1, define stops | Short-lean above FG 75 |
| Frequent Traders | Keep activity, moderate size increase | Maintain frequency, reduce size |
| Infrequent Traders | No change | No change |
| Struggling Traders | Reduce size, wait for confirmation | Sit out |
